In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.utils import bootstrap_project_paths

project_root = bootstrap_project_paths()

DATA_ROOT = project_root / "data/datathon-2026-round-1"


In [ ]:
from pathlib import Path
from src.data.loader import load_orders, load_order_items, load_web_traffic, load_inventory, load_sales
from src.features.build import build_daily_one_row_per_day, drop_columns

root = project_root

orders_df = load_orders(DATA_ROOT)
order_items_df = load_order_items(DATA_ROOT)
web_traffic_df = load_web_traffic(DATA_ROOT)
inventory_df = load_inventory(DATA_ROOT)
sales_df = load_sales(DATA_ROOT)

daily_df = build_daily_one_row_per_day(
    orders_df=orders_df,
    order_items_df=order_items_df,
    web_traffic_df=web_traffic_df,
    inventory_df=inventory_df,
    sales_df=sales_df,
)

daily_df = drop_columns(daily_df, ["same_day_delivery_orders", "weekend_order_share","same_day_delivery_rate", "inv_reorder_products"])

print("shape:", daily_df.shape)
print("duplicated date rows:", int(daily_df["date"].duplicated().sum()))
print("date range:", daily_df["date"].min(), "->", daily_df["date"].max())

print("\ncolumns:")
for i, col in enumerate(daily_df.columns, start=1):
    print(f"{i:02d}. {col}")

from src.utils.plot_features import build_daily, plot_by_year, plot_all_year, plot_overlay

In [ ]:
# Feature Classification
import re
import pandas as pd

all_cols = list(daily_df.columns)

# 1. ID / index
id_cols = ['date']

# 2. Target
target_cols = ['Revenue', 'COGS']

# 3. Temporal
temporal_cols = [c for c in all_cols if c in {
    'year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year',
    'is_weekend', 'is_odd_year', 'is_august_odd', 'is_q3_odd',
    'month_x_odd_year', 'biennial_sin', 'biennial_cos',
    'is_post_2019', 'regime_year_idx', 'days_to_month_end', 'is_month_end_5d',
    'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'doy_sin', 'doy_cos',
}]

# 4. Order & Fulfilment (raw counts / means)
order_raw = {
    'orders_count', 'customers_count', 'returned_orders', 'cancelled_orders',
    'delivered_orders', 'payment_value_total', 'installments_mean',
    'shipping_fee_total', 'shipping_lead_days_mean', 'shipping_lead_days_p90',
    'fulfillment_days_mean', 'payment_value_mean', 'payment_value_std',
    'payment_value_median', 'customer_tenure_days_mean',
    'customer_tenure_days_p90', 'weekend_orders', 'same_day_ship_orders',
    'same_day_delivery_orders', 'return_rate_orders', 'cancel_rate_orders',
    'weekend_order_share', 'same_day_ship_rate', 'same_day_delivery_rate',
    'shipped_orders_count', 'shipped_fee_total', 'delivered_orders_count',
}
order_cols = [c for c in all_cols if c in order_raw]

# 5. Product & Pricing (raw)
product_raw = {
    'items_count', 'quantity_sold', 'gross_merch_value',
    'discount_total', 'net_merch_value', 'list_merch_value',
    'items_cogs_total', 'unique_products_sold', 'unit_price_mean',
    'unit_price_std', 'refund_amount_total', 'return_quantity_total',
    'returned_items_events', 'returned_items_quantity', 'returned_refund_value',
}
product_cols = [c for c in all_cols if c in product_raw]

# 6. Promo (raw counts + type/channel share)
promo_raw = {
    'promo_1_active_lines', 'promo_2_active_lines', 'stackable_promo_lines',
    'promo_1_usage_count', 'promo_2_usage_count',
    'promo_1_discount_value_mean', 'promo_2_discount_value_mean',
    'promo_1_min_order_value_mean', 'promo_2_min_order_value_mean',
}
promo_cols = [c for c in all_cols
              if c in promo_raw
              or re.match(r'promo_(type|channel)_', c)]

# 7. Customer & Demographic share
customer_prefixes = (
    'payment_method_share_', 'device_share_', 'order_source_share_',
    'acquisition_share_', 'gender_share_', 'age_group_share_',
)
customer_cols = [c for c in all_cols if c.startswith(customer_prefixes)]

# 7b. Geography share
geography_cols = [c for c in all_cols
                  if c.startswith('region_share_')
                  or c.startswith('district_share_')]

# 8. Product Mix share
product_mix_cols = [c for c in all_cols
                    if c.startswith('item_category_share_')
                    or c.startswith('item_segment_share_')
                    or c.startswith('item_size_share_')
                    or c.startswith('item_color_share_')]

# 9. Review & Satisfaction
review_raw = {'rating_mean', 'reviews_count', 'reviews_rating_mean', 'reviews_rating_std'}
review_cols = [c for c in all_cols if c in review_raw]

# 10. Web Traffic
web_raw = {
    'sessions', 'unique_visitors', 'page_views', 'bounce_rate',
    'avg_session_duration_sec',
}
web_cols = [c for c in all_cols
            if c in web_raw or c.startswith('traffic_source_share_')]

# 11. Inventory
inventory_cols = [c for c in all_cols if c.startswith('inv_')]

# 12. Engineered Ratios / Composite
engineered_raw = {
    'gross_margin', 'gross_margin_ratio', 'aov_revenue', 'units_per_order',
    'discount_rate', 'effective_discount_rate', 'refund_rate',
    'returns_event_rate', 'returns_quantity_rate', 'refund_to_revenue_rate',
    'pages_per_session', 'sessions_per_visitor', 'pages_per_visitor',
    'revenue_per_session', 'conversion_rate', 'delivery_throughput_rate',
    'checkout_efficiency', 'aov_true', 'avg_item_price', 'basket_size',
    'orders_per_customer', 'product_diversity', 'refund_per_order',
    'return_items_ratio', 'discount_per_order', 'discount_efficiency',
    'stackable_promo_rate', 'demand_supply_ratio', 'inventory_pressure',
    'inventory_turnover_proxy', 'review_coverage_rate', 'returning_users',
    'returning_rate', 'engagement_score', 'quality_sessions',
    'loyal_engagement', 'order_loss_rate', 'satisfaction_demand',
    'promo_intensity', 'promo_active_line_rate', 'engaged_sessions',
    'stock_availability', 'inventory_gap', 'revenue_driver', 'quality_revenue',
    'demand_strength', 'review_sentiment_score', 'revenue_to_cogs_items_gap',
    'orders_per_session', 'revenue_per_visitor', 'items_per_pageview',
    'cogs_per_session', 'customers_per_session',
    'conversion_rate_7d', 'conversion_rate_28d', 'aov_x_orders',
}
engineered_cols = [c for c in all_cols if c in engineered_raw]

# 13. Lag / Rolling / Diff temporal features
_lag_re = re.compile(r'_(lag_|roll_mean_|roll_std_|diff_|pct_change_|yoy_ratio_)')
lag_cols = [c for c in all_cols if _lag_re.search(c)]

# ---- catch-all ----
classified = set(
    id_cols + target_cols + temporal_cols + order_cols + product_cols
    + promo_cols + customer_cols + product_mix_cols + review_cols
    + web_cols + inventory_cols + engineered_cols + lag_cols
    + geography_cols
)
unclassified_cols = [c for c in all_cols if c not in classified]

# ---- summary ----
groups = {
    'ID':                     id_cols,
    'Target':                 target_cols,
    'Temporal':               temporal_cols,
    'Order & Fulfilment':     order_cols,
    'Product & Pricing':      product_cols,
    'Promo':                  promo_cols,
    'Customer & Demographic': customer_cols,
    'Geography':              geography_cols,
    'Product Mix Share':      product_mix_cols,
    'Review & Satisfaction':  review_cols,
    'Web Traffic':            web_cols,
    'Inventory':              inventory_cols,
    'Engineered Ratios':      engineered_cols,
    'Lag / Rolling / Diff':   lag_cols,
    'Unclassified':           unclassified_cols,
}

rows = []
for g, cols in groups.items():
    rows.append({
        'Group':   g,
        'Count':   len(cols),
        'Examples': ', '.join(cols[:4]) + (' ...' if len(cols) > 4 else ''),
    })

summary_df = pd.DataFrame(rows)
print(summary_df.to_string(index=False))
print(f'\nTotal features (excl. ID & target): {len(all_cols) - len(id_cols) - len(target_cols)}')
print(f'Total columns                      : {len(all_cols)}')

## TARGET

In [ ]:
import pandas as pd
fields = [c for c in ['Revenue', 'COGS'] if c in daily_df.columns]
daily = build_daily(daily_df, fields=fields)

sub_df = pd.read_csv('submission_direct.csv')
sub_df['date'] = pd.to_datetime(sub_df['Date'])

sub_cols = ['date'] + fields
daily_merged = pd.concat([daily, sub_df[sub_cols]], axis=0, ignore_index=True)
daily_merged = daily_merged.drop_duplicates(subset=['date'], keep='last')
daily_merged = daily_merged.sort_values('date')

plot_overlay(daily_merged, fields=fields, title_prefix='Target')


### Target (All Years Continuous)

In [ ]:
import pandas as pd
fields = [c for c in ['Revenue', 'COGS'] if c in daily_df.columns]
daily = build_daily(daily_df, fields=fields)

sub_df = pd.read_csv('data\datathon-2026-round-1\sample_submission.csv')
sub_df['date'] = pd.to_datetime(sub_df['Date'])

# Cột date ở daily đang là column chứ không phải index
# Nên ta giữ nguyên date làm column cho sub_df rồi mới concat
sub_cols = ['date'] + fields
daily_merged = pd.concat([daily, sub_df[sub_cols]], axis=0, ignore_index=True)

daily_merged = daily_merged.drop_duplicates(subset=['date'], keep='last')
daily_merged = daily_merged.sort_values('date')

plot_all_year(daily_merged, fields=fields)


In [ ]:
import numpy as np
import pandas as pd

# --- Chuẩn hóa bảng năm ---
regime_df = daily_df.copy()
regime_df['date'] = pd.to_datetime(regime_df['date'], errors='coerce')
regime_df = regime_df.dropna(subset=['date']).sort_values('date')
regime_df['year'] = regime_df['date'].dt.year
regime_df['month'] = regime_df['date'].dt.month

num_cols = regime_df.select_dtypes(include=[np.number]).columns.tolist()
annual = regime_df.groupby('year', as_index=False)[num_cols].mean()

# --- 1) Mô tả regime của target ---
target_cols = [c for c in ['Revenue', 'COGS'] if c in annual.columns]
print('=== Target Regime Summary ===')
if target_cols:
    pre = annual[annual['year'].between(2013, 2018)]
    post = annual[annual['year'].between(2019, 2022)]
    for t in target_cols:
        pre_mean = pre[t].mean() if not pre.empty else np.nan
        post_mean = post[t].mean() if not post.empty else np.nan
        drop_pct = (post_mean / pre_mean - 1.0) * 100 if pre_mean not in [0, np.nan] else np.nan
        print(f'{t}: pre(2013-2018)={pre_mean:,.2f} | post(2019-2022)={post_mean:,.2f} | change={drop_pct:.2f}%')

# --- 2) Tìm tín hiệu gãy + hồi phục ở business features ---
exclude = {'year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year',
           'Revenue', 'COGS'}
candidate_features = [
    c for c in annual.columns
    if c not in exclude and pd.api.types.is_numeric_dtype(annual[c])
]

pre_years = annual[annual['year'].between(2013, 2018)].copy()
post_years = annual[annual['year'].between(2019, 2022)].copy()

rows = []
for col in candidate_features:
    pre_series = pre_years[col].dropna()
    post_series = post_years[col].dropna()
    if len(pre_series) < 3 or len(post_series) < 3:
        continue

    pre_mean = float(pre_series.mean())
    post_mean = float(post_series.mean())
    break_pct = np.nan if abs(pre_mean) < 1e-9 else (post_mean / pre_mean - 1.0)

    # Slope hồi phục trong regime 2019-2022
    x = post_years['year'].to_numpy(dtype=float)
    y = post_years[col].to_numpy(dtype=float)
    if np.all(np.isnan(y)) or np.nanstd(y) < 1e-12:
        post_slope = 0.0
    else:
        msk = ~np.isnan(y)
        post_slope = float(np.polyfit(x[msk], y[msk], 1)[0]) if msk.sum() >= 2 else 0.0

    # Liên hệ với Revenue trong post regime
    if 'Revenue' in post_years.columns:
        rv = post_years['Revenue'].to_numpy(dtype=float)
        msk = ~np.isnan(y) & ~np.isnan(rv)
        corr_post_revenue = float(np.corrcoef(y[msk], rv[msk])[0, 1]) if msk.sum() >= 3 else 0.0
    else:
        corr_post_revenue = 0.0

    impact_score = post_slope * corr_post_revenue
    rows.append({
        'feature': col,
        'break_pct_2019_vs_2013_2018': break_pct,
        'post_slope_2019_2022': post_slope,
        'corr_with_revenue_post': corr_post_revenue,
        'impact_score': impact_score,
    })

signals = pd.DataFrame(rows).sort_values('impact_score', ascending=False)

print('\n=== Top Positive Recovery Signals (favor uplift) ===')
display(signals.head(12))

print('\n=== Top Negative/Headwind Signals ===')
display(signals.sort_values('impact_score', ascending=True).head(12))

# --- 3) Kiểm tra tín hiệu chu kỳ odd/even year (hữu ích cho 2023 là odd year) ---
if {'Revenue', 'COGS'}.issubset(regime_df.columns):
    parity = regime_df.assign(is_odd_year=(regime_df['year'] % 2).astype(int))
    parity_summary = parity.groupby(['is_odd_year', 'month'], as_index=False)[['Revenue', 'COGS']].mean()
    aug_odd = parity_summary[(parity_summary['is_odd_year'] == 1) & (parity_summary['month'] == 8)]
    aug_even = parity_summary[(parity_summary['is_odd_year'] == 0) & (parity_summary['month'] == 8)]

    print('\n=== Odd/Even Year August Signal ===')
    if not aug_odd.empty and not aug_even.empty:
        rev_ratio = float(aug_odd['Revenue'].iloc[0] / max(aug_even['Revenue'].iloc[0], 1e-9))
        cogs_ratio = float(aug_odd['COGS'].iloc[0] / max(aug_even['COGS'].iloc[0], 1e-9))
        print(f'August odd/even Revenue ratio: {rev_ratio:.3f}')
        print(f'August odd/even COGS ratio   : {cogs_ratio:.3f}')
    else:
        print('Không đủ dữ liệu để tính tín hiệu tháng 8 odd/even.')

In [ ]:
# 4) Actionable signals only: loại bỏ feature bám trực tiếp target
def _is_target_proxy(name: str) -> bool:
    s = name.lower()
    return ('revenue' in s) or ('cogs' in s) or ('gross_margin' in s)

actionable = signals[~signals['feature'].map(_is_target_proxy)].copy()

# Chọn tín hiệu có tương quan đủ mạnh với Revenue trong post regime
actionable = actionable[actionable['corr_with_revenue_post'].abs() >= 0.4]

print('=== Actionable positive signals (non-target features) ===')
display(actionable.sort_values('impact_score', ascending=False).head(15))

print('=== Actionable headwinds (non-target features) ===')
display(actionable.sort_values('impact_score', ascending=True).head(15))

## REGIME BREAK SIGNALS (2013-2022)
Phân tích tín hiệu phá xu hướng ổn định để hỗ trợ giả thuyết uplift năm 2023-2024.

## ORDER & FULFILMENT

In [ ]:
fields = [c for c in ['orders_count', 'customers_count', 'returned_orders', 'cancelled_orders', 'delivered_orders', 'payment_value_total', 'installments_mean', 'shipping_fee_total', 'shipping_lead_days_mean', 'shipping_lead_days_p90', 'fulfillment_days_mean', 'payment_value_mean', 'payment_value_std', 'payment_value_median', 'customer_tenure_days_mean', 'customer_tenure_days_p90', 'weekend_orders', 'same_day_ship_orders', 'same_day_delivery_orders', 'return_rate_orders', 'cancel_rate_orders', 'weekend_order_share', 'same_day_ship_rate', 'same_day_delivery_rate', 'shipped_orders_count', 'shipped_fee_total', 'delivered_orders_count'] if c in daily_df.columns]
daily = build_daily(daily_df, fields=fields)
plot_all_year(daily, fields=fields)
# plot_by_year(daily, fields=fields)

## PRODUCT & PRICING

In [ ]:
fields = [c for c in ['items_count', 'quantity_sold', 'gross_merch_value', 'discount_total', 'net_merch_value', 'list_merch_value', 'items_cogs_total', 'unique_products_sold', 'unit_price_mean', 'unit_price_std', 'refund_amount_total', 'return_quantity_total', 'returned_items_events', 'returned_items_quantity', 'returned_refund_value'] if c in daily_df.columns]
daily = build_daily(daily_df, fields=fields)
plot_all_year(daily, fields=fields)
# plot_by_year(daily, fields=fields)

## PROMO

In [ ]:
import re
fields = [c for c in daily_df.columns
          if c in {
              'promo_1_active_lines', 'promo_2_active_lines', 'stackable_promo_lines',
              'promo_1_usage_count', 'promo_2_usage_count',
              'promo_1_discount_value_mean', 'promo_2_discount_value_mean',
              'promo_1_min_order_value_mean', 'promo_2_min_order_value_mean',
          }
          or re.match(r'promo_(type|channel)_', c)]
daily = build_daily(daily_df, fields=fields)
plot_all_year(daily, fields=fields)
# plot_by_year(daily, fields=fields)

## CUSTOMER & DEMOGRAPHIC

In [ ]:
fields = [c for c in daily_df.columns
          if c.startswith((
              'payment_method_share_', 'device_share_', 'order_source_share_',
              'acquisition_share_', 'gender_share_', 'age_group_share_',
          ))]
daily = build_daily(daily_df, fields=fields)
plot_all_year(daily, fields=fields)
# plot_by_year(daily, fields=fields)

## GEOGRAPHY

In [ ]:
fields = [c for c in daily_df.columns
          if c.startswith('region_share_')
          or c.startswith('district_share_')]
daily = build_daily(daily_df, fields=fields)
plot_all_year(daily, fields=fields)
# plot_by_year(daily, fields=fields)

## PRODUCT MIX SHARE

In [ ]:
fields = [c for c in daily_df.columns
          if c.startswith('item_category_share_')
          or c.startswith('item_segment_share_')
          or c.startswith('item_size_share_')
          or c.startswith('item_color_share_')]
daily = build_daily(daily_df, fields=fields)
plot_all_year(daily, fields=fields)
# plot_by_year(daily, fields=fields)

## REVIEW & SATISFACTION

In [ ]:
fields = [c for c in ['rating_mean', 'reviews_count', 'reviews_rating_mean', 'reviews_rating_std'] if c in daily_df.columns]
daily = build_daily(daily_df, fields=fields)
plot_all_year(daily, fields=fields)
# plot_by_year(daily, fields=fields)

## WEB TRAFFIC

In [ ]:
fields = [c for c in daily_df.columns
          if c in {
              'sessions', 'unique_visitors', 'page_views',
              'bounce_rate', 'avg_session_duration_sec',
          }
          or c.startswith('traffic_source_share_')]
daily = build_daily(daily_df, fields=fields)
plot_all_year(daily, fields=fields)
# plot_by_year(daily, fields=fields)

## INVENTORY

In [ ]:
fields = [c for c in daily_df.columns if c.startswith('inv_')]
daily = build_daily(daily_df, fields=fields)
plot_all_year(daily, fields=fields)
# plot_by_year(daily, fields=fields)

## ENGINEERED RATIOS

In [ ]:
all_cols = list(daily_df.columns)
engineered_raw = {
    'gross_margin', 'gross_margin_ratio', 'aov_revenue', 'units_per_order',
    'discount_rate', 'effective_discount_rate', 'refund_rate',
    'returns_event_rate', 'returns_quantity_rate', 'refund_to_revenue_rate',
    'pages_per_session', 'sessions_per_visitor', 'pages_per_visitor',
    'revenue_per_session', 'conversion_rate', 'delivery_throughput_rate',
    'checkout_efficiency', 'aov_true', 'avg_item_price', 'basket_size',
    'orders_per_customer', 'product_diversity', 'refund_per_order',
    'return_items_ratio', 'discount_per_order', 'discount_efficiency',
    'stackable_promo_rate', 'demand_supply_ratio', 'inventory_pressure',
    'inventory_turnover_proxy', 'review_coverage_rate', 'returning_users',
    'returning_rate', 'engagement_score', 'quality_sessions',
    'loyal_engagement', 'order_loss_rate', 'satisfaction_demand',
    'promo_intensity', 'promo_active_line_rate', 'engaged_sessions',
    'stock_availability', 'inventory_gap', 'revenue_driver', 'quality_revenue',
    'demand_strength', 'review_sentiment_score', 'revenue_to_cogs_items_gap',
}
fields = [c for c in all_cols if c in engineered_raw]
daily = build_daily(daily_df, fields=fields)
plot_all_year(daily, fields=fields)
# plot_by_year(daily, fields=fields)